In [11]:
import numpy as np
import mujoco
import numpy as np
from mujoco import viewer
from math import radians
import time

Читаем угловые положения для всех сочленений из сгенерированного файла npy

In [12]:
# Заменить 'имя_файла.npy' на путь к твоему файлу
data_qpos = np.load('qpos_dataset_29052025.npy', allow_pickle=True)

print(data_qpos)

[{'scale': 0.0010000000474974513, 'qpos': {'Joint_pinkie_abduction': 0.5619753003120422, 'Joint_pinkie_PPflexion': 0.5665340423583984, 'Joint_pinkie_DPflexion': 0.28397759795188904, 'Joint_index_abduction': 0.5818096995353699, 'Joint_index_PPflexion': 0.4124312996864319, 'Joint_index_DPflexion': 0.676738440990448, 'Joint_thumb_rotation': 0.043514374643564224, 'Joint_thumb_abduction': 0.18577958643436432, 'Joint_thumb_PPflexion': 0.03315342590212822, 'Joint_thumb_DPflexion': 0.17390210926532745, 'WRJRx': -1.0374289829193928, 'WRJRy': -0.34739138829851435, 'WRJRz': -3.064510099211993, 'WRJTx': 0.11502483487129211, 'WRJTy': 0.08997662365436554, 'WRJTz': -0.1518566757440567}, 'qpos_st': {'Joint_pinkie_abduction': 0.6803157329559326, 'Joint_pinkie_PPflexion': 0.4668615758419037, 'Joint_pinkie_DPflexion': 0.18371979892253876, 'Joint_index_abduction': 0.5125473737716675, 'Joint_index_PPflexion': 0.2779902219772339, 'Joint_index_DPflexion': 0.5420742630958557, 'Joint_thumb_rotation': 0.0438210

In [13]:
# Путь к XML-файлу и qpos
xml_path = "hand_object_edited.xml"
qpos_path = "qpos_dataset_29052025.npy"

# # Загрузка модели и данных
model = mujoco.MjModel.from_xml_path(xml_path)
data = mujoco.MjData(model)

In [26]:
# Загрузка qpos из .npy файла
data_list = np.load(qpos_path, allow_pickle=True)
sample = data_list[4]  # Возьмём первую позу
qpos_dict = sample['qpos']

print(qpos_dict)

{'Joint_pinkie_abduction': -0.4155791103839874, 'Joint_pinkie_PPflexion': 0.6857252717018127, 'Joint_pinkie_DPflexion': 0.19657312333583832, 'Joint_index_abduction': 0.506594717502594, 'Joint_index_PPflexion': 0.23887236416339874, 'Joint_index_DPflexion': 0.6406430006027222, 'Joint_thumb_rotation': 0.1010691374540329, 'Joint_thumb_abduction': 0.03827343508601189, 'Link_thumb_PPflexion': 0.11226855218410492, 'Joint_thumb_DPflexion': 0.010528458282351494, 'WRJRx': 0.7740469941412039, 'WRJRy': 1.0217295461647171, 'WRJRz': 2.754005471762009, 'WRJTx': -0.19956667721271515, 'WRJTy': -0.1569013148546219, 'WRJTz': -0.007115669082850218}


In [27]:
# Создаём массив qpos в правильном порядке
qpos_array = np.zeros(model.nq)
for i in range(model.nq):
    joint_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i)
    print(joint_name)
    if joint_name in qpos_dict:
        qpos_array[i] = qpos_dict[joint_name]
        print(qpos_array[i])
    else:
        print(f"Warning: joint '{joint_name}' not found in qpos_dict — using 0")

print(qpos_array)

Joint_pinkie_abduction
-0.4155791103839874
Joint_pinkie_PPflexion
0.6857252717018127
Joint_pinkie_DPflexion
0.19657312333583832
Joint_index_abduction
0.506594717502594
Joint_index_PPflexion
0.23887236416339874
Joint_index_DPflexion
0.6406430006027222
Joint_thumb_rotation
0.1010691374540329
Joint_thumb_abduction
0.03827343508601189
Joint_thumb_PPflexion
Joint_thumb_DPflexion
0.010528458282351494
[-0.41557911  0.68572527  0.19657312  0.50659472  0.23887236  0.640643
  0.10106914  0.03827344  0.          0.01052846]


## Если меняли углв на радианы еще в XML для джоинтов, то код ниже не применять

In [ ]:
# Transform to radians
for i in range(model.nq):
    joint_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i)
    if joint_name in qpos_dict:
        qpos_array[i] = radians(qpos_dict[joint_name])
        qpos_array[i] = qpos_array[i]

print(qpos_array)

## Continue

In [28]:
# Установка qpos
data.qpos[:] = qpos_array
mujoco.mj_forward(model, data)

In [29]:
# Визуализация
with viewer.launch_passive(model, data) as v:
    print("Press ESC to exit")
    while v.is_running():
        v.sync()

Press ESC to exit


## Последовательность поз

In [ ]:
data_list = np.load("qpos_dataset_29052025.npy", allow_pickle=True)

poses = [entry['qpos'] for entry in data_list]

print(poses)


In [ ]:
# Настройка визуализации (без окна)
renderer = mujoco.Renderer(model)
duration_per_pose = 0.5  # секунд
fps = 30
frames_per_pose = int(duration_per_pose * fps)

In [ ]:
# Функция: обновить позу из словаря
def set_qpos_from_dict(model, data, qpos_dict):
    qpos_array = np.zeros(model.nq)
    for j in range(model.njnt):
        joint_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, j)
        if joint_name in qpos_dict:
            # if 'T' in joint_name:  # трансляция — оставляем как есть
            #     qpos_array[j] = qpos_dict[joint_name]
            # else:  # вращение — переводим в радианы
            #     qpos_array[j] = np.deg2rad(qpos_dict[joint_name])
            qpos_array[j] = qpos_dict[joint_name]
    
    data.qpos[:] = qpos_array
    mujoco.mj_forward(model, data)

In [ ]:
# Запуск визуализации и анимации
with mujoco.viewer.launch_passive(model, data) as viewer:
    while viewer.is_running():
        for pose in poses:
            set_qpos_from_dict(model, data, pose)
            viewer.sync()
            time.sleep(0.5)  # Задержка между позами

## Визуализация руки с капсулями контакта

In [ ]:
# Путь к XML-файлу и qpos
xml_path = "DP-Flex_opened_kinematics_contact_capsules.xml"

# # Загрузка модели и данных
model = mujoco.MjModel.from_xml_path(xml_path)
data = mujoco.MjData(model)


In [ ]:
# Визуализация
with viewer.launch_passive(model, data) as v:
    print("Press ESC to exit")
    while v.is_running():
        v.sync()